## Single stratified 60/20/20 split

Builds a single stratified train/val/test split of the authentic dataset with a fixed 60/20/20 ratio. Class proportions are preserved across the three subsets. This split was used during the exploratory phase of model development.

In [ ]:
import shutil
from pathlib import Path

from sklearn.model_selection import train_test_split


SEED = 42
class_names = ["t72nolabel", "t80nolabel", "t90nolabel"]

SOURCE_DIR = Path(r"F:\Users\nikol\Desktop\Data\Pictures\RealData")
OUTPUT_DIR = Path(r"F:\Users\nikol\Desktop\Data\Pictures\authentic_split")


def create_stratified_split(source_dir, output_dir, seed=42):
    """Build a single stratified 60/20/20 train/val/test split."""
    if output_dir.exists():
        shutil.rmtree(output_dir)

    for split in ["train", "val", "test"]:
        for cls in class_names:
            (output_dir / split / cls).mkdir(parents=True, exist_ok=True)

    summary = {}
    for cls in class_names:
        files = sorted(
            f for f in (source_dir / cls).iterdir()
            if f.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")
        )

        # 60/40 split, then halve the 40% into val and test.
        train_files, temp_files = train_test_split(files, test_size=0.4, random_state=seed)
        val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=seed)

        for f in train_files:
            shutil.copy2(f, output_dir / "train" / cls / f.name)
        for f in val_files:
            shutil.copy2(f, output_dir / "val" / cls / f.name)
        for f in test_files:
            shutil.copy2(f, output_dir / "test" / cls / f.name)

        summary[cls] = {"train": len(train_files), "val": len(val_files), "test": len(test_files)}
        print(f"{cls}: train={len(train_files)}, val={len(val_files)}, test={len(test_files)}")

    return summary


split_summary = create_stratified_split(SOURCE_DIR, OUTPUT_DIR, seed=SEED)
print("\nDone.")